In [ ]:
import numpy as np
import pandas as pd

# Rastgele veri seti oluşturma (Gürültü eklenmiş)
np.random.seed(3)
data_noisy = {
    "ID": range(1, 101),
    #"Gerçek_Değer": np.random.randint(100, 1000, size=100),
    "Ölçüm_1": np.random.randint(100, 1000, size=100) + np.random.normal(0, 50, 100),
    "Ölçüm_2": np.random.randint(100, 1000, size=100) + np.random.normal(0, 100, 100),
    #"Ölçüm_3": np.random.randint(100, 1000, size=100) + np.random.normal(0, 200, 100),
}

df_noisy = pd.DataFrame(data_noisy)
print(df_noisy.head())  # İlk 5 satırı göster


In [ ]:
df_moving_avg = df_noisy.copy()
df_moving_avg["Ölçüm_1_MA"] = df_noisy["Ölçüm_1"].rolling(window=4).mean()
df_moving_avg["Ölçüm_2_MA"] = df_noisy["Ölçüm_2"].rolling(window=4).mean()

print(df_moving_avg.head(10))  # Hareketli ortalama sonrası ilk 10 satırı göster


In [ ]:
bin_count1 = 4  # 4 eşit genişlikli bölme
df_binned1 = df_noisy.copy()
for col in ["Ölçüm_1", "Ölçüm_2"]:
    df_binned1[col] = pd.cut(df_binned1[col], bins=bin_count1, labels=False)  # Eşit genişlikli bölmeleme

df_binned1.head(7)



In [ ]:
bin_count2 = 4
df_binned2 = df_noisy.copy()
for col in ["Ölçüm_1", "Ölçüm_2"]:
    df_binned2[col] = pd.qcut(df_binned2[col], q=bin_count2, labels=False)
    
df_binned2.head(7)

In [ ]:
frekanslar = df_binned2["Ölçüm_1"].value_counts().sort_index()
print(frekanslar)

In [ ]:

from sklearn.linear_model import LinearRegression
from scipy.interpolate import UnivariateSpline

df_fitted = df_noisy.copy()

# Ölçüm_1 için polinom regresyon kullanarak eğri uydurma
X_vals = np.arange(len(df_fitted)).reshape(-1, 1)
y_vals = df_fitted["Ölçüm_1"].values.reshape(-1, 1)

model = LinearRegression()
model.fit(X_vals, y_vals)
df_fitted["Ölçüm_1_Fitted"] = model.predict(X_vals)  # Lineer regresyon ile düzeltilmiş değerler

# Ölçüm_2 için spline interpolation kullanarak eğri uydurma
spline = UnivariateSpline(X_vals.flatten(), df_fitted["Ölçüm_2"], s=500)
df_fitted["Ölçüm_2_Fitted"] = spline(X_vals.flatten())
print(df_fitted.head())  # Eğri uydurma sonrası ilk 5 satırı göster
